## Data processing

1 Setup libraries


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler


1 Merge two dataset into one 

In [ ]:
cust = pd.read_json('../data/customers.json')
trans = pd.read_json('../data/transaction.json')

df = pd.merge(trans,cust, left_on= 'Sender Account ID', right_on = 'Customer ID', how = "left")
df = df.drop(columns = ['Customer ID'])
df = df.fillna(0)

df.to_json('../data/data_2/data.json')


2 Preprocessing


2.1 Extract Features

In [ ]:
if 'Date of Birth' in df.columns:
    df['Date of Birth'] = pd.to_datetime(df['Date of Birth'], errors='coerce')
    df['Age'] = (datetime(2026, 3, 6) - df['Date of Birth']).dt.days // 365
    df = df.drop(columns=['Date of Birth'])

#Time-based flags
if 'DayofWeek' in df.columns:
    df['Is_Weekend'] = df['DayofWeek'].isin([5,6]).astype(int)
if 'Hours' in df.columns:
    df['Is_Night'] = df['Hour'].apply(lambda h: 1 if h >= 22 or h <= 6 else 0)
    
# Ratio Features
if 'Account Balance' in df.columns and 'Salary (per month)' in df.columns:
    salary = df['Salary (per month)'].replace(0, np.nan)
    df['Balance_to_Salary_Ratio'] = df['Account balance'] / salary
if 'Transaction amount' in df.columns and 'Account balance' in df.columns:
    balance = df['Account balance'].replace(0, np.nan)
    df['Transaction_to_Balance_Ratio'] = df['Transaction amount'] / balance

df = df.fillna(0)

2.2 Normalize

In [ ]:
cols_to_scale = [
    'Transaction amount', 'Account balance', 'Salary (per month)',
    'Age', 'Balance_to_Salary_Ratio', 'Transaction_to_Balance_Ratio',
]
available = [c for c in cols_to_scale if c in df.columns]

scaler = MinMaxScaler()
df[available] = scaler.fit_transform(df[available])

3 Encoding

This part will convert text in the dataset into unique numbers which are crucial for models to understand 

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import os

3.1 Utility Functions

We'll keep your core logic here

In [ ]:
def encode_columns(df, columns=None):
    if columns is None:
        columns = df.select_dtypes(include=['object']).columns.tolist()

    encoders = {}
    for col in columns:
        if col in df.columns:
            encoder = LabelEncoder()
            # We cast to string to avoid errors with mixed types or NaNs
            df[col] = encoder.fit_transform(df[col].astype(str))
            encoders[col] = encoder
            print(f"Successfully encoded: {col}")

    return df, encoders

3.2 Configuration & File Paths

In [ ]:
# Paths
INPUT_FILE = '../data/data_2/data.json'
OUTPUT_FILE = '../data/data_2/data_encoded.json'

# Columns to encode (text → numbers)
TEXT_COLUMNS = [
    'Transaction Detail', 'Geological', 'Device Use', 
    'Gender', 'Location', 'Working Status'
]

3.3 Execution & Data Verification



In [ ]:
# 1. Load data
if os.path.exists(INPUT_FILE):
    df = pd.read_json(INPUT_FILE)
    print(f"Loaded {len(df)} rows.")
    
    # 2. Encode
    df_encoded, encoders = encode_columns(df.copy(), TEXT_COLUMNS)
    
    # 3. Save
    df_encoded.to_json(OUTPUT_FILE, orient='records', indent=4, force_ascii=False)
    print(f"Saved encoded data to: {OUTPUT_FILE}")
    
    # 4. Preview the transformation
    display(df_encoded.head())
else:
    print(f"Error: Could not find file at {INPUT_FILE}. Please check your path!")